# Week 3: Penetration Testing

This lab goes over some of the core-concepts of Penetration Testing (Pen Testing). We will use Python to simulate and understand the initial stages if a penetration test, focusing on **reconnaissance** and **vulnerability assessment**.

### Ethics and Legality Notes:

- ALWAYS obtain explicit, written permission before testing ANY system that you do not own.
- Conduct all activities within a designated, isolated test environment (e.g., a local VM, a dedicated test lab, or platforms like HackTheBox/OverTheWire).
- Unauthorized testing is illegal and unethical. The scripts and tools here are for educational purposes only, within your own controlled environment.

### 1. Whois domain lookup

Whois domain lookup allows you to trace the onwnership and tenure of a domain name. All domain name registries maintain a record of information about every domain name purchased through them, along with who owns it, and the date till which it has been purchased.

The Whois database contains details such as the registration date of the domain name, when it expires, ownership and contact information, nameserver information of the domain, the registrar via which the domain was purchased, etc.

In [23]:
import socket
import requests

def get_domain_info(domain):
    try:
        # Get IP address (active but low-risk)
        ip = socket.gethostbyname(domain)
        print(f"\nIP Address: {ip}")

        # Print domain name
        print(f"Domain: {domain}")
        
        # Get public WHOS-like info (passive, using a free API)
        response = requests.get(f"https://ipapi.co/{ip}/json/")
        if response.status_code == 200:
            data = response.json()
            print(f"Organisation: {data.get('org', 'Unknown')}")
            print(f"City: {data.get('city', 'Unknown')}")
            print(f"Country: {data.get('country_name', 'Unknown')}")
        else:
            print("Could not fetch WHOIS data.")
    except Exception as e:
        print(f"Error: {e}")

# Example: use a public domain
for domain in ["python.org", "google-gruyere.appspot.com", "oracle.com", "google.co.uk", "bbc.co.uk", "cdjapan.co.jp"]:
    get_domain_info(domain)


IP Address: 151.101.128.223
Domain: python.org
Could not fetch WHOIS data.

IP Address: 142.250.151.153
Domain: google-gruyere.appspot.com
Could not fetch WHOIS data.

IP Address: 138.1.33.162
Domain: oracle.com
Could not fetch WHOIS data.

IP Address: 142.250.151.94
Domain: google.co.uk
Could not fetch WHOIS data.

IP Address: 151.101.64.81
Domain: bbc.co.uk
Could not fetch WHOIS data.

IP Address: 202.234.167.56
Domain: cdjapan.co.jp
Could not fetch WHOIS data.


For some reason, bbc.co.uk says it's hosted in Montreal, Canada but when I search the location by the IP address, the location it gives is San Francisco, California. This is likely because BBC might have a gateway provider.

# 2. Types of Penetration Tests

Test Basis:
- **Black Box (Opaque):** Minimal info, like external attacker. Realistic for outsiders, shows external posture.
- **White Box (Transparent):** Full access (code, architecture). Thorough, efficient.
**Test Types:**
1. **Bespoke Software:** For custom apps, e.g., web; provides coding feedback.
2. **Scenario-Driven:** Tests risks like lost devices or insider threats.
3. **Detection and Response:** Assesses vulnerabilities plus detection/response. Select based on goals—black box for realism, white box for
depth. Supplements routine security.

The code below simulates a black box on a web page. 

Note: if "Server: Unknown", servers can hide headers for security reasons.

In [ ]:
import requests

def black_box_recon(url):
    try:
        response = requests.get(url)
        print(f"\nBlack Box Findings:")
        print(f"server: {response.headers.get('Server', 'Unknown')}")
        print(f"Content-type: {response.headers.get('Content-Type', 'Unknown')}")
    except Exception as e:
        print(f"Error: {e}")

known_info = {"server": "Apache 2.4", "vulns": "Check CVE-2021-41773"}

for url in ["http://python.org", "http://google-gruyere.appspot.com", "http://oracle.com", "http://google.co.uk", "http://bbc.co.uk", "http://cdjapan.co.jp"]:
    black_box_recon(url)


Black Box Findings:
server: Unknown
Content-type: text/html; charset=utf-8

Black Box Findings:
server: Google Frontend
Content-type: text/html; charset=utf-8

Black Box Findings:
server: AkamaiGHost
Content-type: text/html

Black Box Findings:
server: gws
Content-type: text/html; charset=ISO-8859-1

Black Box Findings:
server: BBC-GTM
Content-type: text/html

Black Box Findings:
server: nginx/1.2.1
Content-type: text/html;charset=UTF-8


# 3. Penetration Testing Methodology and Tools.

**Methodology steps:**
1. **Reconnaissance:** Gather information (passive or active)
2. **Scanning/Enumeration:** Identify hosts/ports/services.
3. **Vulnerability Assessment:** Find weaknesses.
4. **Exploitation:** Attempt breaches.
5. **Post-Exploitation:** Maintain/expand access.
6. **Reporting:** Document findings. 

**Tools:** Nmap (scanning), Nessus (vulns), Metasploit (exploitation),
etc. Get permission always. Trends: AI, cloud testing, ethical
considerations.

The code below is a basic port scanner simulation (localhost only) using pure Python.

In [4]:
import socket

def scan_ports(host, ports):
    open_ports = []
    for port in ports:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        result = sock.connect_ex((host, port))
        if result == 0:
            open_ports.append(port)
        sock.close()
    return open_ports
host = "127.0.0.1"
ports = [80, 443, 22, 8080, 8000]
open_ports = scan_ports(host, ports)
print(f"Open ports on {host}: {open_ports}")

Open ports on 127.0.0.1: [8000]


I used this scanner while running a local web app I am creating for my other module Dynamic Web Applications. 

The app listens on port 8000 and you can see the script has detected the open port.

(Run on PC for open server ports)

# 4. Using Nmap in Python

Nmap ("Network Mapper") is a free and open-source utility used by network administrators and security professionals for **network discovery, security auditing, and inventory management**. It sends raw IP packets and analyses responses to identify active hosts, open ports, operating systems, and services on a network.

**Key Features and Uses**
- **Host Discovery:** Identifies which hosts are online and available on
the network.
- **Port Scanning:** Determines which ports are open on target systems, a key step in identifying potential entry points.
- **Service and Version Detection:** Identifies the application name, version number, and protocol of services listening on ports (e.g., a web server, DNS server).
- **Operating System (OS) Detection:** Uses TCP/IP stack fingerprinting to determine the OS and its version running on a target host.
- **Vulnerability Detection:** The Nmap Scripting Engine (NSE) uses Lua-based scripts to automate a wide range of tasks, including advanced service detection and vulnerability checks.
- **Firewall and IDS Evasion:** Includes various techniques to bypass packet filters and intrusion detection systems.
- **Cross-platform Support:** Runs on major operating systems, including Linux, Windows, and macOS.



In [ ]:
# Setup: Install python-nmap if not already installed
%pip install python-nmap

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nmap

def nmap_scan(host, port_range='1-8000'):
    nm = nmap.PortScanner()
    try:
        nm.scan(host, port_range, arguments='-sV') # -sV for service version detection

        for host in nm.all_hosts():
            print(f"Host: {host} ({nm[host].hostname()})")
            print(f"State: {nm[host].state()}")
        for proto in nm[host].all_protocols():
            print(f"Protocol: {proto}")
            lport = nm[host][proto].keys()
            for port in sorted(lport):
                service = nm[host][proto][port]
                print(f"Port: {port}\tState: {service['state']}\tService:{service.get('name', 'unknown')} {service.get('version', '')}")
    except Exception as e:
        print(f"Error: {e}")

# Example: Scan localhost
nmap_scan('127.0.0.1', '1-10')
print("")
nmap_scan('127.0.0.1', '7950-8000')

Host: 127.0.0.1 (localhost)
State: up
Protocol: tcp
Port: 1	State: closed	Service:tcpmux 
Port: 2	State: closed	Service:compressnet 
Port: 3	State: closed	Service:compressnet 
Port: 4	State: closed	Service: 
Port: 5	State: closed	Service:rje 
Port: 6	State: closed	Service: 
Port: 7	State: closed	Service:echo 
Port: 8	State: closed	Service: 
Port: 9	State: closed	Service:discard 
Port: 10	State: closed	Service: 

Host: 127.0.0.1 (localhost)
State: up
Protocol: tcp
Port: 8000	State: open	Service:http 


I set up the nmap scanner to scan port 8000, since that is where my web app is running. Nmap detected the open port and:
- The host is 127.0.0.1 which is localhost.
- That the connection is running.
- That it is using the TCP protocol.
- And that the port is 8000, state is open and is servicing HTTP.